# LLM-to-WM: generated Python POSG world models

This notebook treats generated Python as the world model, then validates and rolls it out into an explicit state/action/transition representation for graph-focused EDA.

In [ ]:
# CELL 1: Imports
from pathlib import Path
from typing import Any, Dict, List
import re
import matplotlib.pyplot as plt
import networkx as nx
import ollama
from IPython.display import display

from world_model_eda import (
    AbstractPOSG, ExperimentConfig, ExperimentRunner, OllamaPOSGBackend, WorldModelEDA, action_influence, batch_posg_cases, build_state_action_graph,
    build_state_graph, canonical_rollout_json, classify_failure, execute_generated_posg,
    load_reference_game_cases, plot_cross_case_eda, player_action_statistics, results_table,
    rollout_world_model, run_posg_experiment as _run_posg_experiment, summarize_rollout, validate_posg,
)


## Canonical Python POSG interface and validation

`AbstractPOSG`, `validate_posg`, and `rollout_world_model` are imported from `world_model_eda`. The validator exhaustively checks every reachable joint action for small models and returns structured errors and warnings instead of returning `None`.

In [ ]:
# CELL 2: Optional direct inspection of an already-instantiated model
def inspect_world_model(posg: AbstractPOSG, max_states: int = 1_000, max_depth: int = 20):
    validation = validate_posg(posg, max_states=max_states, max_depth=max_depth)
    print(f"passed={validation['passed']}; states={validation['states_checked']}; "
          f"state-action pairs={validation['transitions_checked']}")
    for message in validation['errors']:
        print('ERROR:', message)
    for message in validation['warnings']:
        print('WARNING:', message)
    return validation


In [ ]:
# CELL 3: Ollama generation reusing the experiment runner backend
generation_backend = OllamaPOSGBackend()
def generate_posg_code(game_description: str, model_name: str) -> str:
    return generation_backend(game_description, model_name)


In [ ]:
# CELL 4: Generation -> execution -> validation -> rollout -> graph -> EDA
def run_posg_experiment(game_description: str, model_name: str = 'qwen2.5-coder:latest',
                        max_states: int = 1_000, max_depth: int = 20) -> Dict[str, Any]:
    result = _run_posg_experiment(
        game_description,
        model_name=model_name,
        code_generator=generate_posg_code,
        max_states=max_states,
        max_depth=max_depth,
    )
    if result['success']:
        summary = result['eda_summary']
        print(f"Complete: {summary['states']['state_count']} states, "
              f"{summary['actions']['joint_action_count']} joint actions, "
              f"{summary['transitions']['stochastic_transition_fraction']:.0%} stochastic.")
    else:
        print(f"Pipeline stopped cleanly during {result['stage']}.")
        for error in result['errors']:
            print('ERROR:', error)
    return result


## Explicit rolled-out world model

Successful results contain `rollout`, not just a Python object. It contains reachable states with terminal/depth metadata, legal actions, every joint action, transition outcomes with probabilities, observations, rewards, warnings, and errors.

In [ ]:
# CELL 5: Rollout inspection
def print_rollout(rollout: Dict[str, Any]) -> None:
    print('World Model Summary')
    print('-------------------')
    for key, value in summarize_rollout(rollout).items():
        if key not in {'errors', 'warnings'}: print(f'{key}: {value}')
    print('Players:', rollout['players'])
    print('Initial distribution:', rollout['initial_distribution'])
    print('Reachable states:')
    for state, data in rollout['states'].items():
        print(f"  {state!r}: terminal={data['terminal']}, depth={data['depth']}")
    print('Transitions:')
    for record in rollout['transitions']:
        print(f"  {record['state']!r} -- {record['joint_action']} --> "
              f"{[(o['next_state'], o['probability']) for o in record['outcomes']]}" )

def export_explicit_world_model(rollout: Dict[str, Any], path: str = 'rolled_out_world_model.json') -> str:
    Path(path).write_text(canonical_rollout_json(rollout), encoding='utf-8')
    return path


In [ ]:
# CELL 6: Graph construction
def graph_representations(result: Dict[str, Any]):
    if not result['success']:
        return None, None
    return result['state_graph'], result['state_action_graph']


In [ ]:
# CELL 7: EDA summary and player/action influence
def eda_report(result: Dict[str, Any]) -> Dict[str, Any]:
    if not result['success']:
        return {'success': False, 'errors': result['errors']}
    return {
        'success': True,
        'summary': result['eda_summary'],
        'player_actions': result['player_action_statistics'],
        'action_influence': result['action_influence'],
    }


In [ ]:
# CELL 8: Readable graph-focused visualizations for small models
def plot_experiment(result: Dict[str, Any]) -> None:
    if not result['success']:
        print('No plots: generation, execution, or validation failed.')
        return
    eda = result['eda']
    figure, axes = plt.subplots(2, 2, figsize=(14, 10))
    eda.plot_state_graph(ax=axes[0, 0])
    eda.plot_branching_distribution(ax=axes[0, 1])
    eda.plot_transition_entropy_distribution(ax=axes[1, 0])
    eda.plot_degree_distribution(ax=axes[1, 1])
    figure.tight_layout()

    action_graph = result['state_action_graph']
    if action_graph.number_of_nodes() <= 40:
        plt.figure(figsize=(12, 7))
        positions = nx.spring_layout(action_graph, seed=7)
        colors = ['#f4a261' if data['node_type'] == 'action' else '#457b9d'
                  for _, data in action_graph.nodes(data=True)]
        labels = {node: data.get('label', str(node)) for node, data in action_graph.nodes(data=True)}
        nx.draw_networkx(action_graph, positions, node_color=colors, labels=labels, font_size=8, arrows=True)
        plt.title('State-action-transition graph')
        plt.axis('off')


In [ ]:
# CELL 9: PettingZoo execution adapter (an adapter, not the research representation)
from gymnasium import spaces
from pettingzoo import ParallelEnv
import random

class POSGToPettingZoo(ParallelEnv):
    metadata = {'name': 'generated_posg_v0'}
    def __init__(self, posg: AbstractPOSG):
        self.posg, self.possible_agents = posg, list(posg.players)
        initial = posg.initial_state()
        initial = next(iter(initial)) if isinstance(initial, dict) else initial
        self._actions = {p: posg.legal_actions(initial, p) for p in self.possible_agents}
        self._spaces = {p: spaces.Discrete(len(a)) for p, a in self._actions.items()}
        self._observations = {p: spaces.Text(max_length=512) for p in self.possible_agents}
        self.agents, self.current_state = [], None
    def action_space(self, agent): return self._spaces[agent]
    def observation_space(self, agent): return self._observations[agent]
    def reset(self, seed=None, options=None):
        if seed is not None: random.seed(seed)
        self.agents = self.possible_agents[:]
        initial = self.posg.initial_state()
        self.current_state = next(iter(initial)) if isinstance(initial, dict) else initial
        return {p: 'GAME_START' for p in self.agents}, {p: {} for p in self.agents}
    def step(self, actions):
        joint = {p: self._actions[p][actions[p]] for p in self.agents}
        distribution = self.posg.transition(self.current_state, joint)
        next_state = random.choices(list(distribution), weights=list(distribution.values()), k=1)[0]
        observations = {p: self.posg.observation(self.current_state, joint, next_state, p) for p in self.agents}
        rewards = {p: float(self.posg.reward(self.current_state, joint, next_state, p)) for p in self.agents}
        self.current_state = next_state
        terminal = self.posg.is_terminal(next_state)
        terminations = {p: terminal for p in self.agents}
        truncations, infos = {p: False for p in self.agents}, {p: {} for p in self.agents}
        if terminal: self.agents = []
        return observations, rewards, terminations, truncations, infos


In [ ]:
# CELL 10: G.1 Rock-Paper-Scissors experiment
g1_rps_description = '''Two people sit down to play a single round of rock-paper-scissors.
Both players simultaneously choose rock, paper, or scissors. Rock beats scissors,
scissors beats paper, paper beats rock, and identical choices draw. The round ends immediately.'''

g1_result = run_posg_experiment(g1_rps_description, model_name='qwen2.5-coder:latest')
def check_g1_sanity(result: Dict[str, Any]) -> Dict[str, Any]:
    if not result['success']:
        return {'passed': False, 'reason': f"pipeline failed during {result['stage']}"}
    summary = result['eda_summary']
    checks = {
        'four reachable states': summary['states']['state_count'] == 4,
        'three terminal outcomes': summary['states']['terminal_state_count'] == 3,
        'nine joint actions': summary['actions']['joint_action_count'] == 9,
        'deterministic transitions': summary['transitions']['deterministic_transition_fraction'] == 1.0,
    }
    return {'passed': all(checks.values()), 'checks': checks, 'summary': summary}

if g1_result['success']:
    print_rollout(g1_result['rollout'])
    display(check_g1_sanity(g1_result))
    display(eda_report(g1_result))
    display({'action_influence': g1_result['action_influence']})
    plot_experiment(g1_result)


In [ ]:
# CELL 11: LLM and executable-reference batch runners
def run_game_batch(game_descriptions: Dict[str, str], model_name: str = 'qwen2.5-coder:latest'):
    rows, results = [], {}
    for case, description in game_descriptions.items():
        result = run_posg_experiment(description, model_name=model_name)
        results[case] = result
        if result['success']:
            summary = result['eda_summary']
            rows.append({'case': case, 'success': True, **summary['states'], **summary['transitions'], **summary['graph']})
        else:
            rows.append({'case': case, 'success': False, 'stage': result['stage'], 'errors': result['errors']})
    return rows, results

def run_reference_game_batch(game_case_directory: str = 'data/Test_Cases/03_game'):
    # Integration fixtures execute the repository's expected schemas through AbstractPOSG.
    # They test rollout/EDA across G1-G9; they are not LLM-generated Python results.
    return batch_posg_cases(load_reference_game_cases(game_case_directory), model_name='reference-schema')

reference_g1_g9_rows = run_reference_game_batch()
reference_eda_table = [
    {key: value for key, value in row.items() if key not in {'rollout', 'eda_summary', 'warnings'}}
    for row in reference_g1_g9_rows
]
display(reference_eda_table)
plot_cross_case_eda(reference_g1_g9_rows)

def failure_records(case_results: Dict[str, Dict[str, Any]], model_name: str):
    return [
        {'model': model_name, 'test_case': case, **classify_failure(result)}
        for case, result in case_results.items() if not result['success']
    ]


In [ ]:
# CELL 12: Controlled Qwen compute pilot and future multi-model hook
g9_blotto_description = '''Two commanders simultaneously allocate three soldiers across two battlefields.
Each allocates 0+3, 1+2, 2+1, or 3+0 soldiers. The commander winning more battlefields wins;
equal battlefield wins result in a draw. The engagement ends immediately.'''

def run_compute_pilot(artifact_root: str = 'experiments', profile_memory: bool = True):
    config = ExperimentConfig(
        models=['qwen'], cases={'G1': g1_rps_description, 'G9': g9_blotto_description},
        samples_per_case=1, max_states=10_000, max_depth=20, seed=42,
        save_artifacts=True, artifact_root=Path(artifact_root), profile_memory=profile_memory,
    )
    records = ExperimentRunner(config).run()
    for record in records:
        print(f"\n{'=' * 50}\nModel: {record['model']}\nCase: {record['case']}\nSample: {record['sample']}\n{'=' * 50}")
        print('Status:', record['status'])
        for key in ('generation_seconds', 'execution_seconds', 'validation_seconds', 'rollout_seconds', 'graph_seconds', 'eda_seconds', 'total_seconds', 'peak_memory_mb'):
            print(f"{key}: {record[key]}")
        print('States:', record.get('reachable_states'), 'Edges:', record.get('edges'))
        if record['status'] == 'failed': print('Failure:', record['failure_type'], record['error_message'])
    return results_table(records)

# pilot_results = run_compute_pilot()

# Optional future multi-model comparison. The same situation is reused unchanged.
def compare_models(game_description: str, model_names: List[str]):
    return {model: run_posg_experiment(game_description, model_name=model) for model in model_names}
